# Transformations on Data Frame

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

In [2]:
spark=SparkSession.builder \
  .appName('myApp') \
  .getOrCreate()

In [3]:
data = [("James","","Smith","36636","M",3000),
    ("Michael","Rose","","40288","M",4000),
    ("Robert","","Williams","42114","M",4000),
    ("Maria","Anne","Jones","39192","F",4000),
    ("Jen","Mary","Brown","","F",-1)
  ]

In [4]:
schema=StructType([
    StructField('fname', StringType(), True),
    StructField('mname', StringType(), True),
    StructField('lname', StringType(), True),
    StructField('id', StringType(), True),
    StructField('sex', StringType(), True),
    StructField('salary',IntegerType(), True)
])

In [5]:
df=spark.createDataFrame(data, schema)

In [6]:
df.show()

+-------+-----+--------+-----+---+------+
|  fname|mname|   lname|   id|sex|salary|
+-------+-----+--------+-----+---+------+
|  James|     |   Smith|36636|  M|  3000|
|Michael| Rose|        |40288|  M|  4000|
| Robert|     |Williams|42114|  M|  4000|
|  Maria| Anne|   Jones|39192|  F|  4000|
|    Jen| Mary|   Brown|     |  F|    -1|
+-------+-----+--------+-----+---+------+



1️⃣ Retrieve firstname, middlename, gender and salary

In [ ]:
df.select('fname','mname','sex','salary').show()

+-------+-----+---+------+
|  fname|mname|sex|salary|
+-------+-----+---+------+
|  James|     |  M|  3000|
|Michael| Rose|  M|  4000|
| Robert|     |  M|  4000|
|  Maria| Anne|  F|  4000|
|    Jen| Mary|  F|    -1|
+-------+-----+---+------+





---



2️⃣ Find total salary grouped by gender

In [ ]:
from pyspark.sql.functions import *
df.groupBy('sex').agg(sum('salary')).show()

+---+-----------+
|sex|sum(salary)|
+---+-----------+
|  M|      11000|
|  F|       3999|
+---+-----------+





---



3️⃣ Cast id (string → integer) and salary (integer → float)

In [ ]:
from pyspark.sql.functions import col, when

df_cast = df.withColumn("id",when(col("id") == "", 0).otherwise(col("id")).cast("int")) \
            .withColumn("salary", when(col("salary")<1, 0).otherwise(col("salary").cast("float")))
df_cast.show()

+-------+-----+--------+-----+---+------+
|  fname|mname|   lname|   id|sex|salary|
+-------+-----+--------+-----+---+------+
|  James|     |   Smith|36636|  M|3000.0|
|Michael| Rose|        |40288|  M|4000.0|
| Robert|     |Williams|42114|  M|4000.0|
|  Maria| Anne|   Jones|39192|  F|4000.0|
|    Jen| Mary|   Brown|    0|  F|   0.0|
+-------+-----+--------+-----+---+------+





---



4️⃣ Update salary by adding 100

In [ ]:
df_update_sal=df_cast.withColumn('salary', (col('salary')+100))
df_update_sal.show()

+-------+-----+--------+-----+---+------+
|  fname|mname|   lname|   id|sex|salary|
+-------+-----+--------+-----+---+------+
|  James|     |   Smith|36636|  M|3100.0|
|Michael| Rose|        |40288|  M|4100.0|
| Robert|     |Williams|42114|  M|4100.0|
|  Maria| Anne|   Jones|39192|  F|4100.0|
|    Jen| Mary|   Brown|    0|  F| 100.0|
+-------+-----+--------+-----+---+------+





---



5️⃣ Create new column new_salary = salary * 10

In [ ]:
df_new_sal=df_update_sal.withColumn('new_salary', col('salary')*10)
df_new_sal.show()

+-------+-----+--------+-----+---+------+----------+
|  fname|mname|   lname|   id|sex|salary|new_salary|
+-------+-----+--------+-----+---+------+----------+
|  James|     |   Smith|36636|  M|3100.0|   31000.0|
|Michael| Rose|        |40288|  M|4100.0|   41000.0|
| Robert|     |Williams|42114|  M|4100.0|   41000.0|
|  Maria| Anne|   Jones|39192|  F|4100.0|   41000.0|
|    Jen| Mary|   Brown|    0|  F| 100.0|    1000.0|
+-------+-----+--------+-----+---+------+----------+





---



6️⃣ Create new column new_salary = salary + 105 and cast to integer

In [ ]:
df_new_sal=df_update_sal.withColumn('new_salary', (col('salary')+150).cast("int"))
df_new_sal.show()
df_new_sal.printSchema()

+-------+-----+--------+-----+---+------+----------+
|  fname|mname|   lname|   id|sex|salary|new_salary|
+-------+-----+--------+-----+---+------+----------+
|  James|     |   Smith|36636|  M|3100.0|      3250|
|Michael| Rose|        |40288|  M|4100.0|      4250|
| Robert|     |Williams|42114|  M|4100.0|      4250|
|  Maria| Anne|   Jones|39192|  F|4100.0|      4250|
|    Jen| Mary|   Brown|    0|  F| 100.0|       250|
+-------+-----+--------+-----+---+------+----------+

root
 |-- fname: string (nullable = true)
 |-- mname: string (nullable = true)
 |-- lname: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- sex: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- new_salary: integer (nullable = true)





---



7️⃣ Create column gender = sex and drop gender

In [ ]:
df_new_sal=df_new_sal.withColumn('gender',col('sex')).drop(col('sex'))
df_new_sal.show()

+-------+-----+--------+-----+------+----------+------+
|  fname|mname|   lname|   id|salary|new_salary|gender|
+-------+-----+--------+-----+------+----------+------+
|  James|     |   Smith|36636|3100.0|      3250|     M|
|Michael| Rose|        |40288|4100.0|      4250|     M|
| Robert|     |Williams|42114|4100.0|      4250|     M|
|  Maria| Anne|   Jones|39192|4100.0|      4250|     F|
|    Jen| Mary|   Brown|    0| 100.0|       250|     F|
+-------+-----+--------+-----+------+----------+------+







---





8️⃣ 8- create a new column gender and new_salary --> gender has the sex value , new_salary has 5 number more from the old salary and new_salary type is fload and drop the sex and salary column

In [18]:
from pyspark.sql.functions import col

df_new_cols=df.withColumn('gender' , col('sex')) \
              .withColumn('new_salary', (col('salary')+5).cast("int")) \
              .drop('sex', 'salary')

df_new_cols.show()

+-------+-----+--------+-----+------+----------+
|  fname|mname|   lname|   id|gender|new_salary|
+-------+-----+--------+-----+------+----------+
|  James|     |   Smith|36636|     M|      3005|
|Michael| Rose|        |40288|     M|      4005|
| Robert|     |Williams|42114|     M|      4005|
|  Maria| Anne|   Jones|39192|     F|      4005|
|    Jen| Mary|   Brown|     |     F|         4|
+-------+-----+--------+-----+------+----------+





---



9️⃣ Rename gender column to sex (without dropping)

In [19]:
rename_gender_df=df_new_cols.withColumnRenamed('gender','sex')
rename_gender_df.show()

+-------+-----+--------+-----+---+----------+
|  fname|mname|   lname|   id|sex|new_salary|
+-------+-----+--------+-----+---+----------+
|  James|     |   Smith|36636|  M|      3005|
|Michael| Rose|        |40288|  M|      4005|
| Robert|     |Williams|42114|  M|      4005|
|  Maria| Anne|   Jones|39192|  F|      4005|
|    Jen| Mary|   Brown|     |  F|         4|
+-------+-----+--------+-----+---+----------+





---



🔟 Create column country with default value "USA"

In [24]:
from pyspark.sql.functions import lit
country_df=rename_gender_df.withColumn('country', lit("USA"))
country_df.show()

+-------+-----+--------+-----+---+----------+-------+
|  fname|mname|   lname|   id|sex|new_salary|country|
+-------+-----+--------+-----+---+----------+-------+
|  James|     |   Smith|36636|  M|      3005|    USA|
|Michael| Rose|        |40288|  M|      4005|    USA|
| Robert|     |Williams|42114|  M|      4005|    USA|
|  Maria| Anne|   Jones|39192|  F|      4005|    USA|
|    Jen| Mary|   Brown|     |  F|         4|    USA|
+-------+-----+--------+-----+---+----------+-------+





---



1️⃣1️⃣ Create columns country = "USA" and variable = 5000

In [26]:
from pyspark.sql.functions import lit

country_variable_df=rename_gender_df.withColumn('country', lit("USA")) \
  .withColumn('variable', lit(5000))

country_variable_df.show()

+-------+-----+--------+-----+---+----------+-------+--------+
|  fname|mname|   lname|   id|sex|new_salary|country|variable|
+-------+-----+--------+-----+---+----------+-------+--------+
|  James|     |   Smith|36636|  M|      3005|    USA|    5000|
|Michael| Rose|        |40288|  M|      4005|    USA|    5000|
| Robert|     |Williams|42114|  M|      4005|    USA|    5000|
|  Maria| Anne|   Jones|39192|  F|      4005|    USA|    5000|
|    Jen| Mary|   Brown|     |  F|         4|    USA|    5000|
+-------+-----+--------+-----+---+----------+-------+--------+





---



1️⃣2️⃣ Drop salary column

In [27]:
drop_salary=df.drop('salary')
drop_salary.show()

+-------+-----+--------+-----+---+
|  fname|mname|   lname|   id|sex|
+-------+-----+--------+-----+---+
|  James|     |   Smith|36636|  M|
|Michael| Rose|        |40288|  M|
| Robert|     |Williams|42114|  M|
|  Maria| Anne|   Jones|39192|  F|
|    Jen| Mary|   Brown|     |  F|
+-------+-----+--------+-----+---+



In [28]:
spark.stop()



---

